# AI worker Colab T4 preflight

Run this notebook after selecting a T4 GPU. It is idempotent after a runtime reset: it installs the local backend again, recreates the report directory, and overwrites the report atomically. It never downloads a model URL from an API request.

In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT_ROOT = Path('/content/capstone_project_new')  # change only if the repo is mounted elsewhere
BACKEND = PROJECT_ROOT / 'backend'
REPORT_DIR = Path('/content/ai-worker-reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
assert (BACKEND / 'pyproject.toml').is_file(), 'Clone or mount the repository first.'
os.environ.update({
    'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2',
    'OPENBLAS_NUM_THREADS': '2', 'TOKENIZERS_PARALLELISM': 'false',
})

In [ ]:
# Re-running this editable install is safe after a Colab reset.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(BACKEND)], check=True)

The upstream RaSa repository declares PyTorch 1.9.1, torchvision 0.10.1, Transformers 4.8.1, and timm 0.4.9. Do not force-install that legacy stack into the current Colab runtime. The report records the actual stack and keeps RaSa blocked until its real checkpoint load/inference smoke probe succeeds.

In [ ]:
report = REPORT_DIR / 'colab-t4-preflight.json'
command = [
    sys.executable, str(BACKEND / 'tools' / 'ai_preflight.py'),
    '--registry', str(BACKEND / 'config' / 'models.example.json'),
    '--resource-config', str(BACKEND / 'config' / 'ai_resources.json'),
    '--profile', 'colab_t4', '--workspace', '/content',
    '--output', str(report), '--allow-not-ready',
]
completed = subprocess.run(command, cwd=BACKEND, text=True, capture_output=True)
print(completed.stdout)
assert completed.returncode == 0, completed.stderr
print(f'Report written to {report}')

Expected at AIW-04: CUDA/T4 may pass, while production models remain blocked because real artifacts and model-load probes belong to AIW-10 through AIW-15. Preserve this report, then rerun the same cell after those artifacts/adapters are installed. A final production-ready report must have `ready=true` and include per-model RAM deltas.